In [8]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypedDict):
    draft: str
    feedback: str
    status: str
    revision_count: Annotated[int, operator.add] 

def generate_draft(state: AgentState):
    print("[Writer] Writing initial draft...")
    return {
        : "Here is the v1 article about LangGraph.", 
        : 1, 
        : "pending"
    }

def evaluate_draft(state: AgentState):
    print("[Evaluator] Reviewing draft...")

    if state.get("revision_count", 0) < 3:
        print("   -> Decision: Needs more depth.")
        return {"feedback": "Expand on the core concepts.", "status": "needs_revision"}
    else:
        print("   -> Decision: Perfect!")
        return {"feedback": "Looks great.", "status": "approved"}

def revise_draft(state: AgentState):
    print(f"[Reviser] Updating draft based on feedback: '{state['feedback']}'...")
    new_draft = f"{state['draft']} + (Revision {state['revision_count']})"
    return {
        : new_draft, 
        : 1, 
        : "pending"
    }

def route_after_evaluation(state: AgentState):
    if state.get("status") == "approved":
        print("[Router] Approved! Exiting graph.")
        return END
    else:
        print("[Router] Rejected! Sending back to Reviser.")
        return "revise_draft"

workflow = StateGraph(AgentState)

workflow.add_node("generate_draft", generate_draft)
workflow.add_node("evaluate_draft", evaluate_draft)
workflow.add_node("revise_draft", revise_draft)

workflow.add_edge(START, "generate_draft")
workflow.add_edge("generate_draft", "evaluate_draft")
workflow.add_edge("revise_draft", "evaluate_draft") 

workflow.add_conditional_edges(
    , 
    route_after_evaluation
)

memory = MemorySaver()

app = workflow.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "auto_thread_1"}}

final_state = app.invoke({}, config)
print(f"Final Draft: {final_state['draft']}")
print(f"Total Revisions: {final_state['revision_count']}")


[Writer] Writing initial draft...
[Evaluator] Reviewing draft...
   -> Decision: Needs more depth.
[Router] Rejected! Sending back to Reviser.
[Reviser] Updating draft based on feedback: 'Expand on the core concepts.'...
[Evaluator] Reviewing draft...
   -> Decision: Needs more depth.
[Router] Rejected! Sending back to Reviser.
[Reviser] Updating draft based on feedback: 'Expand on the core concepts.'...
[Evaluator] Reviewing draft...
   -> Decision: Perfect!
[Router] Approved! Exiting graph.
Final Draft: Here is the v1 article about LangGraph. + (Revision 1) + (Revision 2)
Total Revisions: 3
